# eval_external_2023 — frozen pipeline, one-shot external test (TREC 2023 CT)

Applies the **exact** headline pipeline (frozen `R = elig_first-L512`, multi-view + NQS) to the
**blind TREC 2023** corpus, once. NDCG@10 is the generalization number; the 2023 qrels touch only
the final `pytrec_metrics` call — not the pool, the LLM floor, NQS, or any tuning.

**PREREQUISITES — verify on Drive before spending GPU:**
1. `build_corpus_2023.ipynb` re-run so `trec2023/{doc_fulltext_2023.jsonl (per-field), index2docid_2023.txt, topics2023_text.jsonl, qrels2023.txt}` exist.
2. `train_ensemble_full.ipynb` (POOL_TAG='nqs') re-run so `models/ensemble_nqs.txt` + `models/ensemble_nqs_features.json` exist — this is the ensemble behind the 0.5750 TREC22 headline.
3. Component checkpoints reachable via `resolve_ckpt`: clf_R, clf_topic, retriever-v2, Qwen judge, SapBERT.

**One-shot discipline:** decide nothing from the 2023 number; run once and report it. **Confound to state:** 2023 topics are questionnaire-format (domain shift) — a generalization stress test, not a like-for-like TREC22 rerun.

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q lightgbm pytrec_eval rank-bm25 sentence-transformers transformers accelerate datasets tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
import numpy as np, torch, lightgbm as lgb
from tqdm.auto import tqdm
from ctmatch.experiments import (ExperimentConfig, build_bm25, encode_corpus, rrf_fuse,
    llm_expand_query, cross_encoder_scores, relevant_index, resolve_ckpt,
    llm_yesno_scores, llm_prompt, llm_topicality_prompt, topicality_blob, pytrec_metrics)
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
T23 = f'{DATA_ROOT}/trec2023'
cfg = ExperimentConfig(data_root=DATA_ROOT, pool_tag='nqs')   # frozen R + headline pool
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# The persisted ensemble IS the model behind the TREC22 headline — loaded, never refit here.
booster = lgb.Booster(model_file=cfg.path('models/ensemble_nqs.txt'))
FEAT_ORDER = json.load(open(cfg.path('models/ensemble_nqs_features.json')))
print('repr:', cfg.repr_tag(), '| feature order:', FEAT_ORDER)

In [ ]:
# Load the 2023 corpus (per-field records) + topics + qrels DIRECTLY (a different corpus than the
# 2021 one load_corpus/load_eval know about). Everything downstream uses the corpus-agnostic
# ctmatch.experiments representation/scoring functions, so the representation is identical to TREC22.
corpus_ids = [l.strip() for l in open(f'{T23}/index2docid_2023.txt') if l.strip()]
id2fields = {}
for l in open(f'{T23}/doc_fulltext_2023.jsonl'):
    r = json.loads(l); id2fields[r.get('nct_id') or r.get('doc_id')] = r
corpus_fields = [id2fields.get(d, {}) for d in corpus_ids]
topics = {r['topic_id']: r['topic_text'] for r in map(json.loads, open(f'{T23}/topics2023_text.jsonl'))}
rel = {}
for l in open(f'{T23}/qrels2023.txt'):
    t, _, d, r = l.split(); rel.setdefault(t, {})[d] = int(r)
topics = {t: x for t, x in topics.items() if t in rel}   # judged topics only
print(f'2023 corpus {len(corpus_ids):,} | judged topics {len(topics)}')

In [ ]:
# Retrieval indices (cache): BM25 over the whole doc, dense with retriever-v2 on the retrieval repr.
import pickle
from sentence_transformers import SentenceTransformer
BM25_PKL = f'{T23}/bm25_2023.pkl'
EMB_NPY  = f'{T23}/doc_emb_2023_{cfg.retriever_ckpt.split("/")[-1]}.npy'
if os.path.exists(BM25_PKL):
    bm25 = pickle.load(open(BM25_PKL, 'rb'))
else:
    bm25 = build_bm25(corpus_fields, cfg); pickle.dump(bm25, open(BM25_PKL, 'wb'))
if os.path.exists(EMB_NPY):
    doc_emb = np.load(EMB_NPY)
else:
    doc_emb = encode_corpus(corpus_fields, cfg); np.save(EMB_NPY, doc_emb)   # ~1 hr GPU on 2023 corpus
q_enc = SentenceTransformer(resolve_ckpt(cfg, cfg.retriever_ckpt)); q_enc.max_seq_length = cfg.retriever_max_tokens
print('retrieval ready', doc_emb.shape)

In [ ]:
# Load Qwen once (NQS expansion + both judges). NQS expansion cached per topic.
from transformers import AutoTokenizer, AutoModelForCausalLM
qtok = AutoTokenizer.from_pretrained(resolve_ckpt(cfg, cfg.llm_ckpt), padding_side='left')
if qtok.pad_token is None: qtok.pad_token = qtok.eos_token
qwen = AutoModelForCausalLM.from_pretrained(resolve_ckpt(cfg, cfg.llm_ckpt),
                                            torch_dtype=torch.float16, device_map='auto').eval()

EXP = f'{T23}/nqs_expansions_2023.jsonl'; expansion = {}
if os.path.exists(EXP):
    for l in open(EXP):
        r = json.loads(l); expansion[r['topic_id']] = r['expansion']
else:
    with open(EXP, 'w') as f:
        for t in tqdm(topics, desc='nqs expand'):
            e = llm_expand_query(qwen, qtok, topics[t], cfg); expansion[t] = e
            f.write(json.dumps({'topic_id': t, 'expansion': e}) + '\n')
print('expansions ready:', len(expansion))

In [ ]:
# NQS candidate pool + retrieval features (mirrors nqs_retrieval.retrieve exactly; NQS query = topic + ' ' + expansion).
def retrieve(qtext, k=cfg.cand_k):
    sc = bm25.get_scores(qtext.lower().split()); bt = np.argpartition(-sc, k)[:k]; bt = bt[np.argsort(-sc[bt])]
    qv = q_enc.encode([qtext], normalize_embeddings=True)[0].astype('float32'); sims = doc_emb @ qv
    dt = np.argpartition(-sims, k)[:k]; dt = dt[np.argsort(-sims[dt])]
    bm = {corpus_ids[i]: float(sc[i]) for i in bt}; dn = {corpus_ids[i]: float(sims[i]) for i in dt}
    br = {d: r for r, d in enumerate(bm)}; dr = {d: r for r, d in enumerate(dn)}
    rrf = rrf_fuse([list(br), list(dr)], k=cfg.rrf_k)
    cand = sorted(set(bm) | set(dn), key=lambda d: rrf.get(d, 0), reverse=True)
    feats = {d: {'bm25': bm.get(d, 0.), 'dense': dn.get(d, 0.), 'rrf': rrf.get(d, 0.),
                 'bm25_rank': br.get(d, k), 'dense_rank': dr.get(d, k)} for d in cand}
    return cand, feats

pool, rfeat = {}, {}
for t in tqdm(topics, desc='nqs retrieve'):
    cand, feats = retrieve(topics[t] + ' ' + expansion[t])
    pool[t] = cand
    for d in cand: rfeat[(t, d)] = feats[d]
print('pool built | mean cand/topic', int(np.mean([len(v) for v in pool.values()])))

In [ ]:
# Both LLM judges over the top-500 by RRF (docs[:llm_top_k]) — same pool slice as the feature notebooks.
llm_yesno, topicality = {}, {}
for t in tqdm(topics, desc='judges'):
    top = [d for d in pool[t][:cfg.llm_top_k] if d in id2fields]; ff = [id2fields[d] for d in top]
    for d, s in zip(top, llm_yesno_scores(qwen, qtok, topics[t], ff, cfg, batch=8, prompt_fn=llm_prompt)):
        llm_yesno[(t, d)] = s
    for d, s in zip(top, llm_yesno_scores(qwen, qtok, topics[t], ff, cfg, batch=8, prompt_fn=llm_topicality_prompt)):
        topicality[(t, d)] = s
del qwen; torch.cuda.empty_cache()
print('judges done')

In [ ]:
# Cross-encoder views over the WHOLE pool: clf_R (elig_first) + clf_topic (topic_first).
from transformers import AutoTokenizer as AT, AutoModelForSequenceClassification as ASC
def ce_scores(ckpt, ce_cfg):
    rk = resolve_ckpt(cfg, ckpt); tk = AT.from_pretrained(rk)
    m = ASC.from_pretrained(rk).to(device).eval()
    ridx = relevant_index(m)
    pidx = next((int(i) for i, v in m.config.id2label.items() if 'partial' in str(v).lower()), None)
    R, P = {}, {}
    for t in tqdm(topics, desc=ckpt.split('/')[-1]):
        docs = [d for d in pool[t] if d in id2fields]; ff = [id2fields[d] for d in docs]
        rr = cross_encoder_scores(m, tk, topics[t], ff, ce_cfg, ridx)
        pp = cross_encoder_scores(m, tk, topics[t], ff, ce_cfg, pidx) if pidx is not None else [0.] * len(docs)
        for d, a, b in zip(docs, rr, pp): R[(t, d)] = a; P[(t, d)] = b
    del m; torch.cuda.empty_cache(); return R, P
clf_rel, clf_partial = ce_scores(cfg.clf_ckpt, cfg)
clf_topic_rel, clf_topic_partial = ce_scores(cfg.clf_topic_ckpt, cfg.with_(repr_strategy='topic_first'))
print('cross-encoders done')

In [ ]:
# condition_match_exp: SapBERT cosine(expanded query, doc topicality blob) over the whole pool.
sap = SentenceTransformer(resolve_ckpt(cfg, cfg.topicality_encoder))
uniq = sorted({d for docs in pool.values() for d in docs if d in id2fields})
demb = sap.encode([topicality_blob(id2fields[d], cfg) for d in uniq],
                  normalize_embeddings=True, batch_size=128, show_progress_bar=True).astype('float32')
didx = {d: i for i, d in enumerate(uniq)}
tids = list(topics)
qv = sap.encode([topics[t] + '. ' + expansion.get(t, '') for t in tids],   # expanded query (cm_exp)
                normalize_embeddings=True, batch_size=128).astype('float32')
condition_match = {}
for t, v in zip(tids, qv):
    for d in pool[t]:
        if d in didx: condition_match[(t, d)] = float(v @ demb[didx[d]])
print('condition_match_exp done')

In [ ]:
# Assemble feature vectors in the PERSISTED column order, predict once with the frozen booster.
def featvec(t, d):
    rf = rfeat.get((t, d), {})
    v = {'bm25': rf.get('bm25', 0.), 'bm25_rank': rf.get('bm25_rank', cfg.cand_k),
         'dense': rf.get('dense', 0.), 'dense_rank': rf.get('dense_rank', cfg.cand_k), 'rrf': rf.get('rrf', 0.),
         'clf_rel': clf_rel.get((t, d), 0.), 'clf_partial': clf_partial.get((t, d), 0.),
         'clf_topic_rel': clf_topic_rel.get((t, d), 0.), 'clf_topic_partial': clf_topic_partial.get((t, d), 0.),
         'llm_yesno': llm_yesno.get((t, d), cfg.llm_floor),
         'topicality': topicality.get((t, d), 0.),
         'condition_match': condition_match.get((t, d), 0.)}
    return [v[f] for f in FEAT_ORDER]
run = {}
for t in topics:
    docs = [d for d in pool[t] if d in id2fields]
    X = np.array([featvec(t, d) for d in docs], dtype=np.float32)
    run[t] = {d: float(s) for d, s in zip(docs, booster.predict(X))}
print('scored', len(run), 'topics')

In [ ]:
# One-shot metrics — qrels used ONLY here. NDCG@10 graded; P@10/MRR eligible-only (TREC overview basis).
import pytrec_eval
qrels = {t: {d: int(r) for d, r in rel[t].items()} for t in run}
metrics = pytrec_metrics(run, qrels, k=10)
per = pytrec_eval.RelevanceEvaluator(qrels, {'ndcg_cut.10'}).evaluate(run)
vals = np.array([per[t]['ndcg_cut_10'] for t in per])
boot = [np.mean(np.random.default_rng(i).choice(vals, len(vals), replace=True)) for i in range(10000)]
print('=== TREC 2023 (external, frozen pipeline) — ONE-SHOT ===')
print(metrics)
print(f'NDCG@10 = {vals.mean():.4f}  95% CI [{np.percentile(boot, 2.5):.4f}, {np.percentile(boot, 97.5):.4f}]  (n={len(vals)})')
print('Reference: TREC22 headline 0.5750. Fill the TREC23 best-run NDCG@10 from the 2023 CT overview for context.')